# Clustering practice — tiny arrays (numpy only)

How to use: each Ex has a question cell (`# your code here`,
checks print `Not yet` until filled) followed by a solution cell
(`# SOLUTION - try yourself first`). Tiny 3x2 / 4x2 arrays only,
no heavy plots or datasets (keeps VS Code stable).
Kernel: my-venv (python3). No loops where the project forbids them.

> Master level projects and context: These projects are the hardest
> ones and code-wise often require "tricks" that we may not be
> familiar with.
>
> The difficulty lies here with these projects; it's either a lecture
> on statistics of the method, then you see only math/matrix notation
> and watch videos to hope someone explains it intuitively enough,
> then its implementation in scipy is straightforward with a
> one-liner. To go deeper one must write the algorithm inside the
> method, which then brings the understanding requirement much
> higher, at the level of the dimensions of matrices and numpy tricks
> (because we have use numpy). and since we have been working with
> numpy for only a short time, the code is difficult :D

## Student note — axis disappears (kept)

> axis in numpy reductions – took me way too long to get it right
> whether axis=0 or axis=1 is right and which axis I needed to have
> "disappear"
>
> AXES: 0 for cols, 1 for rows, 2 depth, 3 ... ?

Condensed snippet this note refers to (drilled in Ex1–Ex2):

```python
d = X.shape[1]
mins = X.min(axis=0)  # shape (d,) min of each column
maxs = X.max(axis=0)  # shape (d,) max of each column
```

Rule of thumb: the listed axis is the one that disappears.

## Student note — broadcasting to (n, k, d) (kept)

> Broadcasting into 3D – going from two 2D arrays to a (n, k, d)
> tensor was a bit difficult, also when past 2 dimensions "rows and
> columns" stops working as a mental picture for me.
>
> Tensor == numpy.array (for now :D)
> Dimensions all are arrays (in lin algebra sense) but "tensor" is
> also an object type for a different package (tensorflow). For numpy
> consider multi dim array = np.array. --> simplest way to look at
> things (tensors as math definition has "better" properties for ML)
>
> For now learning about the tips and tricks of np.array s is good
> enough!

Condensed snippet this note refers to (drilled in Ex3–Ex5):

```python
"""use broadcasting trick to make the substraction"""
diffs = X[:, np.newaxis, :] - C
# euclidian distance to all centroids
dists = np.sqrt((diffs ** 2).sum(axis=2))
# find which cluster point belongs to
clss = dists.argmin(axis=1)
# sum of the smallest squared distance across clusters
var = (diffs ** 2).sum(axis=2).min(axis=1).sum()
```

## Student note — Task 6 EM + Jupyter stability (kept)

> Task 6: Expectation Maximization
>
> For plotting I previously kept using Jupyter in VS Code. But it
> kept crashing on me once the datasets/plots got heavier. Maybe
> someone found something more stable to use they can recommend?

This notebook avoids that crash: numpy only, tiny arrays, no plots,
no datasets. Ex7 drills the E-step gap (hard vs soft), Ex8 the M-step.

## Ex1 — axis disappears (easy)

Goal: feel which axis vanishes in `min` / `mean`.
Setup: `X` is (3, 2). Predict shapes BEFORE running.
To do: fill `axis=?` so `a0` is (2,), `a1` is (3,), `m0` is (2,).
Pitfall: `axis=0` collapses rows (per-column); `axis=1` collapses
columns (per-row). The listed axis is the one that disappears.
Usage order: reduce -> `print(shape)` -> compare shapes/values.
Source: numpy docs `ndarray.min` / `ndarray.mean`; RESOURCES.md
Lavrenko K-means; ai-book-kb Ch.9 K-means intro.

In [ ]:
import numpy as np

X = np.array([[1., 3.], [2., 4.], [5., 0.]])

# your code here: choose axis so one axis disappears (no loops)
a0 = None  # TODO: min over rows -> shape (2,)
a1 = None  # TODO: min over cols -> shape (3,)
m0 = None  # TODO: mean over rows -> shape (2,)

print("a0 shape:", getattr(a0, "shape", None))
print("a1 shape:", getattr(a1, "shape", None))
print("m0 shape:", getattr(m0, "shape", None))

try:
    assert a0 is not None, "fill a0 first"
    assert a1 is not None, "fill a1 first"
    assert m0 is not None, "fill m0 first"
    assert a0.shape == (2,), "a0 shape"
    assert a1.shape == (3,), "a1 shape"
    assert m0.shape == (2,), "m0 shape"
    assert np.allclose(a0, [1., 0.]), "a0 values"
    assert np.allclose(a1, [1., 2., 0.]), "a1 values"
    assert np.allclose(m0, [8. / 3., 7. / 3.]), "m0"
    print("Ex1 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([[1., 3.], [2., 4.], [5., 0.]])

# axis=0 collapses rows -> per-column; axis=1 collapses cols
a0 = X.min(axis=0)
print("a0 shape:", a0.shape)
a1 = X.min(axis=1)
print("a1 shape:", a1.shape)
m0 = X.mean(axis=0)
print("m0 shape:", m0.shape)

assert a0.shape == (2,), "a0 shape"
assert a1.shape == (3,), "a1 shape"
assert m0.shape == (2,), "m0 shape"
assert np.allclose(a0, [1., 0.]), "a0 values"
assert np.allclose(a1, [1., 2., 0.]), "a1 values"
assert np.allclose(m0, [8. / 3., 7. / 3.]), "m0"
print("Ex1 checks passed")
# Source: numpy docs ndarray.min/mean (axis); RESOURCES.md
# Lavrenko K-means; ai-book-kb Ch.9 K-means intro

## Ex2 — per-column mins/maxs + one uniform draw (easy)

Goal: Task 0 init pattern — bounds per dimension, one draw, no loop.
Setup: same (3, 2) `X`, `k = 2`, `d = 2`. Seed 0 for hand-check.
To do: `mins = X.min(axis=0)`, `maxs = X.max(axis=0)`, then one
`np.random.uniform(mins, maxs, size=(k, d))` call for `C` (2, 2).
Pitfall: `size=(k, d)` not `(d,)`; bounds broadcast per column.
Usage order: bounds -> seed -> uniform -> `print(shape)` -> bounds.
Source: numpy docs `random.uniform` + broadcasting; Task 0;
RESOURCES.md Lavrenko K-means.

In [ ]:
import numpy as np

X = np.array([[1., 3.], [2., 4.], [5., 0.]])
k, d = 2, X.shape[1]

# your code here: one bound per column, one uniform call, no loops
mins = None  # TODO: X.min(axis=0) -> shape (2,)
maxs = None  # TODO: X.max(axis=0) -> shape (2,)
np.random.seed(0)
C = None  # TODO: uniform(mins, maxs, size=(k, d))

print("mins shape:", getattr(mins, "shape", None))
print("maxs shape:", getattr(maxs, "shape", None))
print("C shape:", getattr(C, "shape", None))

try:
    assert mins is not None, "fill mins first"
    assert maxs is not None, "fill maxs first"
    assert C is not None, "fill C first"
    assert mins.shape == (2,), "mins shape"
    assert maxs.shape == (2,), "maxs shape"
    assert C.shape == (2, 2), "C shape"
    assert np.allclose(mins, [1., 0.]), "mins"
    assert np.allclose(maxs, [5., 4.]), "maxs"
    assert np.all(C >= mins), "C below mins"
    assert np.all(C <= maxs), "C above maxs"
    print("Ex2 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([[1., 3.], [2., 4.], [5., 0.]])
k, d = 2, X.shape[1]

# per-column bounds broadcast into one uniform draw, no loops
mins = X.min(axis=0)
print("mins shape:", mins.shape)
maxs = X.max(axis=0)
print("maxs shape:", maxs.shape)
np.random.seed(0)
C = np.random.uniform(mins, maxs, size=(k, d))
print("C shape:", C.shape)

assert mins.shape == (2,), "mins shape"
assert maxs.shape == (2,), "maxs shape"
assert C.shape == (2, 2), "C shape"
assert np.allclose(mins, [1., 0.]), "mins"
assert np.allclose(maxs, [5., 4.]), "maxs"
assert np.all(C >= mins), "C below mins"
assert np.all(C <= maxs), "C above maxs"
print("Ex2 checks passed")
# Source: numpy docs random.uniform + broadcasting; Task 0

## Ex3 — newaxis shapes (medium)

Goal: predict the (n, k, d) broadcast before running it.
Setup: `X` (3, 2), `C` (2, 2), so `n=3, k=2, d=2`.
To do: `Xe = X[:, None, :]` then `diffs = Xe - C`; state in words
what `diffs[i, j]` is (answer: `X[i] - C[j]`, a (d,) vector).
Pitfall: `X[:, None, :]` is (3, 1, 2), not (3, 2); `C` broadcasts
over axis 0 of that (n, k, d) tensor.
Usage order: newaxis -> `print(shape)` -> subtract ->
`print(shape)` -> read one `diffs[i, j]`.
Source: numpy docs broadcasting + `newaxis`; Task 1; RESOURCES.md
Lavrenko K-means (assign step).

In [ ]:
import numpy as np

X = np.array([[0., 0.], [1., 1.], [2., 0.]])
C = np.array([[0., 0.], [1., 0.]])

# your code here: build the (n, k, d) tensor, no loops
# diffs[i, j] in words: vector from centroid j to point i
Xe = None  # TODO: X[:, None, :] -> shape (3, 1, 2)
diffs = None  # TODO: Xe - C -> shape (3, 2, 2)

print("Xe shape:", getattr(Xe, "shape", None))
print("diffs shape:", getattr(diffs, "shape", None))

try:
    assert Xe is not None, "fill Xe first"
    assert diffs is not None, "fill diffs first"
    assert Xe.shape == (3, 1, 2), "Xe shape"
    assert diffs.shape == (3, 2, 2), "diffs shape"
    assert np.allclose(diffs[0, 1], [-1., 0.]), "d00"
    assert np.allclose(diffs[2, 0], [2., 0.]), "d20"
    print("Ex3 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([[0., 0.], [1., 1.], [2., 0.]])
C = np.array([[0., 0.], [1., 0.]])

# (3, 2) -> (3, 1, 2); minus (2, 2) broadcasts to (3, 2, 2)
# diffs[i, j] = X[i] - C[j], the vector from centroid j to point i
Xe = X[:, None, :]
print("Xe shape:", Xe.shape)
diffs = Xe - C
print("diffs shape:", diffs.shape)

assert Xe.shape == (3, 1, 2), "Xe shape"
assert diffs.shape == (3, 2, 2), "diffs shape"
assert np.allclose(diffs[0, 1], [-1., 0.]), "d00"
assert np.allclose(diffs[2, 0], [2., 0.]), "d20"
print("Ex3 checks passed")
# Source: numpy docs broadcasting/newaxis; Task 1

## Ex4 — distances + assignment (medium)

Goal: chain `(diffs**2).sum(2) -> sqrt -> argmin(1)` by hand.
Setup: same `X` (3, 2), `C` (2, 2) as Ex3; expect
`sq = [[0, 1], [2, 1], [4, 1]]`, `clss = [0, 1, 1]`.
To do: `sq`, `dists = np.sqrt(sq)`, `clss = dists.argmin(axis=1)`.
Pitfall: `sum(axis=2)` kills `d`; `argmin(axis=1)` kills `k`;
`argmin` on `sq` equals `argmin` on `dists` (sqrt is monotone).
Usage order: square -> sum(2) -> sqrt -> argmin(1), print each.
Source: ai-book-kb Ch.9 pp.263-265, 289 (K-means assign); Task 1;
numpy docs `sum` / `argmin`.

In [ ]:
import numpy as np

X = np.array([[0., 0.], [1., 1.], [2., 0.]])
C = np.array([[0., 0.], [1., 0.]])
diffs = X[:, None, :] - C

# your code here: distances then nearest index, no loops
sq = None  # TODO: (diffs ** 2).sum(axis=2) -> (3, 2)
dists = None  # TODO: np.sqrt(sq) -> (3, 2)
clss = None  # TODO: dists.argmin(axis=1) -> (3,)

print("sq shape:", getattr(sq, "shape", None))
print("dists shape:", getattr(dists, "shape", None))
print("clss shape:", getattr(clss, "shape", None))

try:
    assert sq is not None, "fill sq first"
    assert dists is not None, "fill dists"
    assert clss is not None, "fill clss"
    assert sq.shape == (3, 2), "sq shape"
    assert dists.shape == (3, 2), "dists shape"
    assert clss.shape == (3,), "clss shape"
    assert np.allclose(sq, [[0., 1.], [2., 1.], [4., 1.]]), "sq"
    assert np.allclose(clss, [0, 1, 1]), "clss"
    print("Ex4 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([[0., 0.], [1., 1.], [2., 0.]])
C = np.array([[0., 0.], [1., 0.]])
diffs = X[:, None, :] - C

# sum(2) kills d -> (3, 2); argmin(1) kills k -> (3,)
sq = (diffs ** 2).sum(axis=2)
print("sq shape:", sq.shape)
dists = np.sqrt(sq)
print("dists shape:", dists.shape)
clss = dists.argmin(axis=1)
print("clss shape:", clss.shape)

assert sq.shape == (3, 2), "sq shape"
assert dists.shape == (3, 2), "dists shape"
assert clss.shape == (3,), "clss shape"
assert np.allclose(sq, [[0., 1.], [2., 1.], [4., 1.]]), "sq"
assert np.allclose(clss, [0, 1, 1]), "clss"
print("Ex4 checks passed")
# Source: ai-book-kb Ch.9 pp.263-265, 289; Task 1

## Ex5 — variance one-liner (medium)

Goal: total intra-cluster variance without `sqrt`, without loops.
Setup: same `sq` as Ex4; expected `var = 0 + 1 + 1 = 2.0`.
To do: `var = sq.min(axis=1).sum()` and explain: `min` picks the
nearest centroid per point, `sum` totals over points.
Pitfall: variance uses SQUARED distances — no `sqrt` here.
Usage order: `sum(2)` -> `min(1)` -> `sum()`; print scalar last.
Source: ai-book-kb Ch.9 pp.263-265, 289 (WCSS objective); Task 2.

In [ ]:
import numpy as np

sq = np.array([[0., 1.], [2., 1.], [4., 1.]])

# your code here: nearest per point, then total (no loops, no sqrt)
# min = nearest centroid; sum = total over points
var = None  # TODO: sq.min(axis=1).sum() -> scalar 2.0

print("var:", var)

try:
    assert var is not None, "fill var first"
    assert np.allclose(var, 2.0), "var"
    print("Ex5 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

sq = np.array([[0., 1.], [2., 1.], [4., 1.]])

# min(1) = nearest centroid per point; sum() = total over points
# squared distances only: no sqrt for variance
var = sq.min(axis=1).sum()
print("var:", var)

assert np.allclose(var, 2.0), "var"
print("Ex5 checks passed")
# Source: ai-book-kb Ch.9 pp.263-265, 289; Task 2

## Ex6 — mask-mean + empty cluster (medium-hard)

Goal: recompute a centroid from its members; spot an empty cluster.
Setup: `X` (4, 2), `clss = [0, 0, 1, 1]`; expect `m0 = [0, 0.5]`,
`m1 = [5.5, 5]`. Cluster 2 is empty by construction.
To do: `m0 = X[clss == 0].mean(axis=0)`; same for `m1`; flag
`is_empty = ((clss == 2).sum() == 0)`; re-seed empty ones with one
`np.random.uniform(mins, maxs)` draw.
Pitfall: `X[mask]` keeps (m, d); `mean(axis=0)` gives (d,). Empty
mask -> `nan`, so check the count BEFORE taking the mean.
Usage order: mask -> `print(shape)` -> mean(0) -> count check.
Source: numpy docs boolean indexing; Task 1; RESOURCES.md Lavrenko
K-means (centroid = mean of members).

In [ ]:
import numpy as np

X = np.array([[0., 0.], [0., 1.], [5., 5.], [6., 5.]])
clss = np.array([0, 0, 1, 1])

# your code here: boolean-mask means, then empty check, no loops
m0 = None  # TODO: X[clss == 0].mean(axis=0) -> (2,)
m1 = None  # TODO: X[clss == 1].mean(axis=0) -> (2,)
# when to re-seed: count is 0 -> uniform draw (see solution)
is_empty = None  # TODO: (clss == 2).sum() == 0 -> True

print("m0 shape:", getattr(m0, "shape", None))
print("m1 shape:", getattr(m1, "shape", None))
print("is_empty:", is_empty)

try:
    assert m0 is not None, "fill m0 first"
    assert m1 is not None, "fill m1 first"
    assert is_empty is not None, "fill flag"
    assert m0.shape == (2,), "m0 shape"
    assert m1.shape == (2,), "m1 shape"
    assert np.allclose(m0, [0., 0.5]), "m0"
    assert np.allclose(m1, [5.5, 5.]), "m1"
    assert is_empty, "flag empty"
    print("Ex6 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([[0., 0.], [0., 1.], [5., 5.], [6., 5.]])
clss = np.array([0, 0, 1, 1])

# mask (m, d) -> mean(axis=0) (d,); check count before mean
m0 = X[clss == 0].mean(axis=0)
print("m0 shape:", m0.shape)
m1 = X[clss == 1].mean(axis=0)
print("m1 shape:", m1.shape)
is_empty = ((clss == 2).sum() == 0)
print("is_empty:", is_empty)
# re-seed pattern when empty: uniform(mins, maxs) (one draw)
mins = X.min(axis=0)
print("mins shape:", mins.shape)
maxs = X.max(axis=0)
print("maxs shape:", maxs.shape)
np.random.seed(1)
reseed = np.random.uniform(mins, maxs)
print("reseed shape:", reseed.shape)

assert m0.shape == (2,), "m0 shape"
assert m1.shape == (2,), "m1 shape"
assert np.allclose(m0, [0., 0.5]), "m0"
assert np.allclose(m1, [5.5, 5.]), "m1"
assert is_empty, "flag empty"
assert reseed.shape == (2,), "reseed"
assert np.all(reseed >= mins), "low"
assert np.all(reseed <= maxs), "high"
print("Ex6 checks passed")
# Source: numpy docs boolean indexing; Task 1

## Ex7 — hard vs soft E-step (hard)

Goal: feel the EM gap — hard picks one cluster, soft splits mass.
Setup: 1D toy, `n=3, k=2`. Likelihood rows are clusters:
`L = [[0.4, 0.3, 0.01], [0.1, 0.1, 0.4]]` (k, n), `pi = [0.5, 0.5]`.
To do: `unnorm = pi[:, None] * L`; `denom = unnorm.sum(axis=0)`;
`g = unnorm / denom` (k, n); `ll = log(denom).sum()`; hard picks
`L.argmax(axis=0) = [0, 0, 1]`.
Pitfall: `g` columns sum to 1 (posterior per point); `ll` logs the
EVIDENCE `denom`, never `log(g)` (that sums to `log 1 = 0`).
Usage order: weight -> evidence -> normalize -> log-sum.
Source: ai-book-kb Ch.9 pp.283-284, 288-289, 292 (EM E-step);
Task 6; RESOURCES.md Lavrenko EM + Brilliant GMM.

In [ ]:
import numpy as np

pi = np.array([0.5, 0.5])
L = np.array([[0.4, 0.3, 0.01], [0.1, 0.1, 0.4]])

# your code here: Bayes posterior per point (one loop max; none here)
unnorm = None  # TODO: pi[:, None] * L -> (2, 3)
denom = None  # TODO: unnorm.sum(axis=0) -> (3,)
g = None  # TODO: unnorm / denom -> (2, 3)
ll = None  # TODO: np.log(denom).sum() -> scalar
hard = None  # TODO: L.argmax(axis=0) -> (3,)

print("g shape:", getattr(g, "shape", None))
print("denom shape:", getattr(denom, "shape", None))
print("ll:", ll)

try:
    assert g is not None, "fill g first"
    assert denom is not None, "fill denom"
    assert ll is not None, "fill ll"
    assert hard is not None, "fill hard"
    assert g.shape == (2, 3), "g shape"
    assert np.allclose(g.sum(axis=0), [1., 1., 1.]), "cols"
    assert np.allclose(g[0, 0], 0.8), "g00"
    assert np.allclose(g[1, 2], 0.9756, atol=1e-4), "g12"
    assert np.allclose(ll, -4.5804, atol=1e-4), "ll"
    assert np.allclose(hard, [0, 0, 1]), "hard"
    print("Ex7 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

pi = np.array([0.5, 0.5])
L = np.array([[0.4, 0.3, 0.01], [0.1, 0.1, 0.4]])

# posterior = weighted likelihood / evidence; ll logs evidence
unnorm = pi[:, None] * L
print("unnorm shape:", unnorm.shape)
denom = unnorm.sum(axis=0)
print("denom shape:", denom.shape)
g = unnorm / denom
print("g shape:", g.shape)
ll = np.log(denom).sum()
print("ll:", ll)
hard = L.argmax(axis=0)
print("hard shape:", hard.shape)

assert g.shape == (2, 3), "g shape"
assert np.allclose(g.sum(axis=0), [1., 1., 1.]), "cols"
assert np.allclose(g[0, 0], 0.8), "g00"
assert np.allclose(g[1, 2], 0.9756, atol=1e-4), "g12"
assert np.allclose(ll, -4.5804, atol=1e-4), "ll"
assert np.allclose(hard, [0, 0, 1]), "hard"
print("Ex7 checks passed")
# Source: ai-book-kb Ch.9 pp.283-284, 288-289, 292; Task 6

## Ex8 — weighted mean M-step, 1D only (hard)

Goal: M-step = familiar stats with soft counts as weights.
Setup: 1D `X = [0, 1, 10]` (3,), reuse Ex7 `g` (2, 3).
To do: `nk = g.sum(axis=1)` (soft counts); `pi = nk / n`;
`m = (g @ X) / nk` (weighted mean). Covariance note only:
1D `s2 = (g * (X - m[:, None])**2).sum(axis=1) / nk`.
Pitfall: `g @ X` is (k,); divide by `nk`, not `n`. `pi` sums to 1.
Usage order: soft count -> prior -> weighted mean -> print each.
Source: ai-book-kb Ch.9 pp.283-284, 288-289, 292 (EM M-step);
Task 7; RESOURCES.md Lavrenko mixture 4 (weighted mean).

In [ ]:
import numpy as np

X = np.array([0., 1., 10.])
g = np.array([[0.8, 0.75, 0.02439], [0.2, 0.25, 0.97561]])
n = X.shape[0]

# your code here: soft stats replace hard counts (no loops)
nk = None  # TODO: g.sum(axis=1) -> (2,)
pi = None  # TODO: nk / n -> (2,)
m = None  # TODO: (g @ X) / nk -> (2,)

print("nk shape:", getattr(nk, "shape", None))
print("pi shape:", getattr(pi, "shape", None))
print("m shape:", getattr(m, "shape", None))

try:
    assert nk is not None, "fill nk first"
    assert pi is not None, "fill pi"
    assert m is not None, "fill m"
    assert nk.shape == (2,), "nk shape"
    assert pi.shape == (2,), "pi shape"
    assert m.shape == (2,), "m shape"
    assert np.allclose(pi.sum(), 1.0), "pi sum"
    assert np.allclose(m[0], 0.631, atol=1e-3), "m0"
    assert np.allclose(m[1], 7.018, atol=1e-3), "m1"
    print("Ex8 checks passed")
except AssertionError as err:
    print("Not yet:", err)

In [ ]:
# SOLUTION - try yourself first
import numpy as np

X = np.array([0., 1., 10.])
g = np.array([[0.8, 0.75, 0.02439], [0.2, 0.25, 0.97561]])
n = X.shape[0]

# soft count -> prior -> weighted mean (divide by nk, not n)
nk = g.sum(axis=1)
print("nk shape:", nk.shape)
pi = nk / n
print("pi shape:", pi.shape)
m = (g @ X) / nk
print("m shape:", m.shape)
# 1D var note only: (g * (X - m[:, None])**2).sum(1) / nk

assert nk.shape == (2,), "nk shape"
assert pi.shape == (2,), "pi shape"
assert m.shape == (2,), "m shape"
assert np.allclose(pi.sum(), 1.0), "pi sum"
assert np.allclose(m[0], 0.631, atol=1e-3), "m0"
assert np.allclose(m[1], 7.018, atol=1e-3), "m1"
print("Ex8 checks passed")
# Source: ai-book-kb Ch.9 pp.283-284, 288-289, 292; Task 7